In [1]:
import os
import json
import re
import unicodedata
import zlib
import torch
import torch.nn as nn
import pyiwn
import gradio as gr

# ==========================================
# 1. FIXED OPEN OVERRIDE (CRASH PREVENTION)
# ==========================================
import builtins
import io

orig_open = io.open

def safe_utf8_open(file, mode="r", *args, **kwargs):
    # Only inject encoding if opening in text mode and not explicitly provided
    if "b" not in mode and "encoding" not in kwargs and len(args) < 2:
        kwargs["encoding"] = "utf-8"
    return orig_open(file, mode, *args, **kwargs)

# Overwrite builtins cleanly without breaking underlying ASGI/FastAPI engines
builtins.open = safe_utf8_open

# ==========================================
# 2. CONFIGURATION & STATE LOADING
# ==========================================
CFG = dict(
    window=5,
    n_buckets=2**12,
    ngram_min=3,
    ngram_max=4,
    max_ctx_feats=256,
    max_gloss_feats=256,
    emb_dim=32,
    hidden=32,
    dropout=0.5,
    max_candidates=12
)

# Download IndoWordNet resources securely if missing
pyiwn.download()
iwn = pyiwn.IndoWordNet(pyiwn.Language.KANNADA)
SYNSETS = list(iwn.all_synsets())

LEMMA_RAW = {unicodedata.normalize("NFC", str(l)): l for s in SYNSETS for l in s.lemma_names()}
LEMMA_VOCAB = set(LEMMA_RAW)

_syn_cache = {}
def get_synsets(lemma):
    if lemma not in LEMMA_VOCAB:
        return []
    if lemma not in _syn_cache:
        seen, out = set(), []
        for s in iwn.synsets(LEMMA_RAW[lemma]):
            if s.synset_id() not in seen:
                seen.add(s.synset_id())
                out.append(s)
        _syn_cache[lemma] = out
    return _syn_cache[lemma]

# ==========================================
# 3. TEXT PROCESSING & LEMMATIZER
# ==========================================
DEP_VOWELS = set("ಾಿೀುೂೃೆೇೈೊೋೌ")
NON_STARTERS = DEP_VOWELS | set("್ಂಃ")
TOKEN_RE = re.compile(r"[\u0C80-\u0CFF]+|[A-Za-z0-9]+")

def clean(text):
    text = unicodedata.normalize("NFC", str(text))
    return re. sub("[\u200b\u200c\u200d]", "", text).strip()

def tokenize(text):
    return TOKEN_RE.findall(clean(text))

class IWNLemmatizer:
    def __init__(self, vocab, min_len=2):
        self.vocab, self.min_len, self._cache = vocab, min_len, {}
    def __call__(self, tok):
        if tok not in self._cache:
            self._cache[tok] = self._lookup(tok)
        return self._cache[tok]
    def _lookup(self, tok):
        for end in range(len(tok), self.min_len - 1, -1):
            stem, nxt = tok[:end], tok[end:end+1]
            if stem[-1] == "್" or nxt in ("್", "ಂ", "ಃ"):
                continue
            options = [stem] + ([stem + "ು"] if nxt in DEP_VOWELS else [])
            for cand in options:
                if cand in self.vocab:
                    return cand, True
        return tok, False

lem = IWNLemmatizer(LEMMA_VOCAB)

def prefix_match(tok, lemma):
    ends_u = lemma.endswith("ು")
    stem = lemma[:-1] if ends_u else lemma
    if len(stem) < 2 or not tok.startswith(stem):
        return False
    rest = tok[len(stem):]
    if rest and rest[0] in NON_STARTERS:
        return ends_u and rest[0] in DEP_VOWELS
    return True

def find_target(tokens, target, lemma):
    for i, t in enumerate(tokens):
        if t == target: return i, "surface"
    for i, t in enumerate(tokens):
        if lem(t)[0] == lemma: return i, "lemmatizer"
    for i, t in enumerate(tokens):
        if prefix_match(t, lemma): return i, "prefix"
    return None, "not_found"

def window(tokens, i, w):
    return tokens[max(0, i - w): i], tokens[i + 1: i + 1 + w]

# ==========================================
# 4. FEATURE ENGINEERING & EMBEDDING
# ==========================================
def stable_hash(s, n_buckets):
    return 1 + zlib.crc32(s.encode("utf-8")) % (n_buckets - 1)

def word_feats(tok):
    lemma, in_wn = lem(tok)
    feats = ["w=" + tok]
    if in_wn and lemma != tok:
        feats.append("l=" + lemma)
    padded = f"<{tok}>"
    for n in range(CFG["ngram_min"], CFG["ngram_max"] + 1):
        feats += [padded[i: i + n] for i in range(len(padded) - n + 1)]
    return feats

def ctx_features(left, right):
    feats = []
    if left: feats.append("L1=" + left[-1])
    if right: feats.append("R1=" + right[0])
    for tok in left + right:
        feats += word_feats(tok)
    return feats

def gloss_features(syn):
    toks = tokenize(syn.gloss()) + [t for l in syn.lemma_names() for t in tokenize(l)]
    feats = []
    for t in toks:
        feats += word_feats(t)
    return feats

def encode(feats, max_len):
    ids = [stable_hash(f, CFG["n_buckets"]) for f in feats][:max_len]
    return ids + [0] * (max_len - len(ids))

# ==========================================
# 5. NEURAL NETWORK MODEL STRUCTURE
# ==========================================
class SenseRanker(nn.Module):
    def __init__(self, n_buckets, dim, hidden, dropout):
        super().__init__()
        self.emb = nn.EmbeddingBag(n_buckets, dim, mode="mean", padding_idx=0)
        def tower():
            return nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(dim, hidden),
                nn.Tanh(),
                nn.Dropout(dropout),
                nn.Linear(hidden, hidden)
            )
        self.ctx_tower, self.gloss_tower = tower(), tower()
        
    def forward(self, ctx, gloss, mask):
        B, k, g = gloss.shape
        c = self.ctx_tower(self.emb(ctx))
        s = self.gloss_tower(self.emb(gloss.reshape(B * k, g))).reshape(B, k, -1)
        scores = torch.einsum("bh,bkh->bk", c, s)
        return scores.masked_fill(~mask, -1e9)

# Load existing model parameters
model = SenseRanker(CFG["n_buckets"], CFG["emb_dim"], CFG["hidden"], CFG["dropout"])

# Match model weights dynamically from your wsd_kn_model directory
MODEL_WEIGHTS_PATH = os.path.join("wsd_kn_model", "ranker.pt")
if os.path.exists(MODEL_WEIGHTS_PATH):
    model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=torch.device('cpu')))
model.eval()

# Build Static Global Gloss Mapping Table
WEAK_SYNSETS = {s.synset_id(): s for s in SYNSETS if s.examples()}
SYN_ROW = {sid: i + 1 for i, sid in enumerate(sorted(WEAK_SYNSETS))}
GLOSS_TABLE = torch.zeros(len(SYN_ROW) + 1, CFG["max_gloss_feats"], dtype=torch.long)
GLOSS_TABLE[0, 0] = 1
for sid, r in SYN_ROW.items():
    GLOSS_TABLE[r] = torch.tensor(encode(gloss_features(WEAK_SYNSETS[sid]), CFG["max_gloss_feats"]))

# ==========================================
# 6. INFERENCE PREDICTION LOGIC
# ==========================================
@torch.no_grad()
def predict_sense(sentence, target_word, lemma=None, top_k=3):
    target_word = clean(target_word)
    lemma = clean(lemma) if lemma else lem(target_word)[0]
    tokens = tokenize(sentence)
    i, method = find_target(tokens, target_word, lemma)
    if i is None:
        raise ValueError(f"'{target_word}' (lemma '{lemma}') not found in sentence.")
    left, right = window(tokens, i, CFG["window"])
    if not (left or right):
        raise ValueError("The sentence has no context words besides the target.")
    cands = get_synsets(lemma)
    if not cands:
        raise ValueError(f"'{lemma}' is not in Kannada IndoWordNet")
    
    senses = [dict(id=s.synset_id(), pos=s.pos(), gloss=s.gloss()) for s in cands]
    if len(cands) == 1:
        return dict(target=target_word, lemma=lemma, left=left, right=right, candidates=senses,
                    ranked=[dict(**senses[0], prob=1.0)], note="only one sense in IndoWordNet")
                    
    ctx = torch.tensor([encode(ctx_features(left, right), CFG["max_ctx_feats"])])
    
    # Secure row indexing checks dynamically
    gloss_list = []
    for s in cands:
        if s.synset_id() in SYN_ROW:
            gloss_list.append(GLOSS_TABLE[SYN_ROW[s.synset_id()]].tolist())
        else:
            gloss_list.append(encode(gloss_features(s), CFG["max_gloss_feats"]))
            
    gloss = torch.tensor([gloss_list])
    probs = model(ctx, gloss, torch.ones(1, len(cands), dtype=torch.bool)).softmax(-1)[0]
    order = probs.argsort(descending=True)[:top_k].tolist()
    return dict(target=target_word, lemma=lemma, left=left, right=right, candidates=senses,
                ranked=[dict(**senses[j], prob=round(float(probs[j]), 3)) for j in order], note="")


2026-10-27:20:52:20,17 INFO     [utils.py:160] NumExpr defaulting to 4 threads.
2026-10-27:20:52:38,48 INFO     [iwn.py:43] Loading kannada language synsets...
